# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice and Why

For the Refresh / Content Opportunity Scoring lane, I selected the **Random Forest** model.

Random Forest is suitable because it can capture complex, non-linear relationships between multiple search performance signals such as impressions, CTR, engagement, and content age. It is robust to noisy data, requires minimal preprocessing, and provides feature importance scores that help explain which features influence the recommendations. It also performs better than simple rule-based approaches when multiple factors interact.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [5]:
!git clone https://github.com/sanjanabhat846/sanju.git
%cd /content/sanju
!ls

Cloning into 'sanju'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 128 (delta 44), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.85 MiB | 10.06 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/sanju
AGENTS.md  DATA_USE.md	LICENSE    README.md	     scripts   submission
CLAUDE.md  docs		notebooks  requirements.txt  SETUP.md  work
data	   GUIDE.md	outputs    sanju	     skills


## Split Design

The dataset is divided into training and testing sets using a grouped split based on **client_id**.

Using grouped validation prevents pages from the same client appearing in both training and testing datasets. This provides a more realistic evaluation because the model is tested on unseen clients rather than memorizing client-specific patterns.

The same split is used for both the Week 4 baseline and the machine learning model to ensure a fair comparison.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create a simple opportunity score
# Higher impressions + lower CTR = higher refresh opportunity
df["opportunity_score"] = (
    df["impressions_90d"] * (1 - df["ctr"] / 100)
)

# Features
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "engagement_rate"
]

X = df[features]
y = df["opportunity_score"]

groups = df["client_id"]

# Grouped split
gss = GroupShuffleSplit(test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)

print("Random Forest MAE:", mae)

Random Forest MAE: 29.941837779301256


## Model vs Baseline

The Random Forest model is trained using the same dataset, features, and train/test split used by the baseline model.

The comparison is performed using the same evaluation metric to ensure a fair assessment of performance.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

The Random Forest model performed better than the baseline on the chosen evaluation metric while using the same train/test split.

Most prediction errors occurred for pages with limited historical search data or low impressions. These pages provide fewer useful signals, making them harder to prioritize accurately.

Feature importance indicates that impressions, CTR, and engagement were the strongest contributors to the model's predictions. The model should be used as a decision-support tool rather than an automatic decision maker.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.